# Various experiments for GNN

In [53]:
from pathlib import Path
from typing import Any, Dict, List, Optional
import json
import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

In [54]:
NUC2ID = { 'A':1, 'C':2, 'G':3, 'T':4, 'N':5 }
def seq_to_tokens(seq: str) -> torch.LongTensor:
    return torch.tensor([NUC2ID.get(c, 5) for c in seq], dtype=torch.long)

def _safe_get(d: Dict[str, Any], key: str, default):
    v = d.get(key, default)
    return default if v is None else v

ID2NUC = {v:k for k,v in NUC2ID.items()}

def tokens_to_seq(tokens: torch.LongTensor) -> str:
    return "".join(ID2NUC.get(int(t), "N") for t in tokens.tolist())

In [55]:
class HyperbubbleDataset(Dataset):
    """
    Minimal GNN-ready dataset from your JSONL:
      - Node features:
          seq_tokens : [N, K] Long
          x_cov      : [N, 1] Float (KC coverage; 0 if unknown)
      - Graph:
          edge_index : [2, E] Long
          edge_attr  : [E, 5] Float (len_nodes, len_bp, cov_min, cov_mean, on_ref)
          start_idx, end_idx : Long
      - Labels:
          y_edge_mask    : [E] Long (1 on labeled path edges, else 0)
          label_path_idx : Long (-1 if none)
    """
    def __init__(self, files: List[str]):
        self.files = [Path(p) for p in files]
        self.records: List[Dict[str, Any]] = []
        for fp in self.files:
            with fp.open("r") as fh:
                for line in fh:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        self.records.append(json.loads(line))
                    except json.JSONDecodeError:
                        continue

    def __len__(self) -> int:
        return len(self.records)

    def _build_graph(self, rec: Dict[str, Any]) -> Data:
        # --- nodes (map seq -> idx), carry best-known coverage ---
        node_seqs: Dict[str, int] = {}
        node_covs: List[float] = []

        def ensure_node(seq: str, cov: Optional[float] = None) -> int:
            if seq not in node_seqs:
                node_seqs[seq] = len(node_seqs)
                node_covs.append(float(cov) if cov is not None else 0.0)
            else:
                i = node_seqs[seq]
                if cov is not None and node_covs[i] == 0.0:
                    node_covs[i] = float(cov)
            return node_seqs[seq]

        # endpoints
        start_seq = rec["start_seq"]
        end_seq   = rec["end_seq"]

        # core nodes
        for n in _safe_get(rec, "nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))

        # 1-hop context
        for n in _safe_get(rec, "upstream_nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))
        for n in _safe_get(rec, "downstream_nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))

        # ---- Strategy-A selected interior nodes (if present) ----
        for n in _safe_get(rec, "inside_nodes", []):
            if isinstance(n, dict) and "seq" in n:
                ensure_node(n["seq"], n.get("cov", 0))

        # make sure endpoints are present
        ensure_node(start_seq)
        ensure_node(end_seq)

        # --- edges + attributes ---
        edge_src, edge_dst, edge_attr = [], [], []
        for e in _safe_get(rec, "edges", []):
            u = ensure_node(e["source_seq"])
            v = ensure_node(e["target_seq"])
            edge_src.append(u)
            edge_dst.append(v)
            edge_attr.append([
                float(e.get("len_nodes", 0)),
                float(e.get("len_bp", 0)),
                float(e.get("cov_min", 0)),
                float(e.get("cov_mean", 0.0)),
                1.0 if e.get("on_ref", False) else 0.0
            ])

        # --- node features ---
        node_order = [None] * len(node_seqs)
        for s, idx in node_seqs.items():
            node_order[idx] = s

        # keep tokens; embedding happens in GNN
        seq_tokens = torch.stack([seq_to_tokens(s) for s in node_order], dim=0)  # [N, K] Long
        x_cov = torch.tensor(node_covs, dtype=torch.float32).unsqueeze(1)        # [N, 1] Float

        start_idx = torch.tensor(node_seqs[start_seq], dtype=torch.long)
        end_idx   = torch.tensor(node_seqs[end_seq], dtype=torch.long)

        edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
        edge_attr  = (torch.tensor(edge_attr, dtype=torch.float32)
                      if edge_attr else torch.zeros((0,5), dtype=torch.float32))

        # --- labels from label_path ---
        num_edges = edge_index.size(1)
        y_edge_mask = torch.zeros(num_edges, dtype=torch.long)
        label_path_idx = -1
        paths_list = _safe_get(rec, "paths", [])
        lp = rec.get("label_path")
        if lp is not None and 0 <= lp < len(paths_list) and num_edges > 0:
            label_path_idx = int(lp)
            y_edge_mask[torch.tensor(paths_list[label_path_idx], dtype=torch.long)] = 1

        data = Data(
            seq_tokens=seq_tokens,                 # [N, K] Long (to be embedded in model)
            x_cov=x_cov,                           # [N, 1] Float
            edge_index=edge_index,                 # [2, E] Long
            edge_attr=edge_attr,                   # [E, 5] Float
            start_idx=start_idx,                   # Long
            end_idx=end_idx,                       # Long
            num_nodes=seq_tokens.size(0),
            y_edge_mask=y_edge_mask,               # [E] Long
            label_path_idx=torch.tensor(label_path_idx, dtype=torch.long),
        )
        if "bubble_id" in rec:
            data.bubble_id = torch.tensor(rec["bubble_id"], dtype=torch.long)
        if "k" in rec:
            data.k = torch.tensor(rec["k"], dtype=torch.long)
        return data

    def __getitem__(self, idx: int) -> Data:
        return self._build_graph(self.records[idx])


In [56]:
from pathlib import Path
from torch.utils.data import Subset
from torch_geometric.loader import DataLoader
import torch

# Always load BOTH
TRAIN_PATHS = [
        "lower_cov_data/ecoli_dataset_cov20_full_upgraded_ratio_lt_35.jsonl",

    "lower_cov_data/klebsiella_dataset_cov20_full_upgraded_ratio_lt_35.jsonl",
]
TEST_PATHS = [
     "lower_cov_data/salmonella_dataset_cov20_full_upgraded_ratio_lt_35.jsonl",
]

# Keep only labeled samples
def labeled_subset(ds):
    labeled_indices = []
    for i in range(len(ds)):
        d = ds[i]
        lp = int(d.label_path_idx.item()) if hasattr(d, "label_path_idx") else -1
        if lp >= 0:
            labeled_indices.append(i)
    return Subset(ds, labeled_indices)


In [57]:
USE_DIRECTML = True

import torch
device = torch.device('cpu')
if not USE_DIRECTML and torch.cuda.is_available():
    device = torch.device('cuda')
elif USE_DIRECTML:
    try:
        import torch_directml
        device = torch_directml.device()
        print("Using DirectML:", device)
    except Exception as e:
        print("DirectML not available, falling back to CPU:", e)
        device = torch.device('cpu')

import torch.nn as nn
import torch.nn.functional as F

class DenseGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim)

    def forward(self, H, edge_index_local, n_nodes: int):
        if n_nodes == 0:
            return H
        A = H.new_zeros((n_nodes, n_nodes))
        if edge_index_local.numel() > 0:
            src, dst = edge_index_local
            one = torch.ones_like(src, dtype=H.dtype)
            A.index_put_((src, dst), one, accumulate=True)

        # add self-loops w/o torch.eye
        idx = torch.arange(n_nodes, device=H.device)
        try:
            A[idx, idx] += 1
        except Exception:
            flat = A.view(-1)
            diag_idx = torch.arange(0, n_nodes*n_nodes, step=n_nodes+1, device=H.device)
            flat.index_add_(0, diag_idx, torch.ones(n_nodes, device=H.device, dtype=H.dtype))
            A = flat.view(n_nodes, n_nodes)

        deg = A.sum(dim=1) + 1e-8
        D_inv_sqrt = deg.pow(-0.5)
        A_norm = (D_inv_sqrt[:, None] * A) * D_inv_sqrt[None, :]
        return A_norm @ self.lin(H)

# ---- Simple GNN with sequence embedding + GCN + edge MLP ----
class HyperbubbleGNN(nn.Module):
    def __init__(
        self,
        vocab_size=5,          # tokens: 0=PAD, A,C,G,T (and optionally N)
        embed_dim=32,          # per-token embedding dim
        gcn_hidden=64,         # node hidden dim (consistent throughout)
        edge_hidden=64,        # hidden in edge MLP
        edge_feat_dim=5,       # [len_nodes, len_bp, cov_min, cov_mean, on_ref]
        use_lstm=False,        # optional: token encoder = mean or BiLSTM
    ):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.use_lstm = use_lstm
        if use_lstm:
            self.lstm = nn.LSTM(embed_dim, gcn_hidden // 2, batch_first=True, bidirectional=False)
            self.node_in = gcn_hidden + 1   # + coverage scalar
        else:
            self.node_in = embed_dim + 1

        self.gcn_hidden = gcn_hidden
        self.gcn1 = DenseGCNLayer(self.node_in, gcn_hidden)
        self.gcn2 = DenseGCNLayer(gcn_hidden, gcn_hidden)

        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * gcn_hidden + edge_feat_dim, edge_hidden),
            nn.ReLU(),
            nn.Linear(edge_hidden, 1),
        )

    def encode_nodes(self, seq_tokens, x_cov):
        """
        seq_tokens: [N, K] tokenized k-mer
        x_cov     : [N, 1]
        returns   : [N, node_in]
        """
        E = self.embed(seq_tokens)
        mask = (seq_tokens != 0).float().unsqueeze(-1)
        if self.use_lstm:
            lengths = mask.squeeze(-1).sum(dim=1).clamp(min=1).long()
            mean_init = (E * mask).sum(dim=1) / lengths.clamp(min=1).unsqueeze(-1).float()
            H0, _ = self.lstm(E)
            Hseq = (H0 * mask).sum(dim=1) / lengths.clamp(min=1).unsqueeze(-1).float()
        else:
            lengths = mask.squeeze(-1).sum(dim=1).clamp(min=1)
            Hseq = (E * mask).sum(dim=1) / lengths.unsqueeze(-1)

        X = torch.cat([Hseq, x_cov], dim=1)
        return X

    def forward(self, data):
        """
        data: PyG batch with fields:
          - seq_tokens [N,K], x_cov [N,1], edge_index [2,E], edge_attr [E,5],
            batch [N] (graph ids), num_nodes, etc.
        returns:
          logits [E]  (edge-wise)
        """
        device = data.seq_tokens.device
        N = data.num_nodes
        E = data.edge_index.size(1)

        X0 = self.encode_nodes(data.seq_tokens, data.x_cov)

        out_node = X0.new_zeros((N, self.gcn_hidden))
        batch_vec = data.batch if hasattr(data, "batch") else torch.zeros(N, dtype=torch.long, device=device)
        num_graphs = int(batch_vec.max().item()) + 1 if N > 0 else 0

        for g in range(num_graphs):
            node_ids = (batch_vec == g).nonzero(as_tuple=False).view(-1)
            n_nodes = int(node_ids.numel())
            if n_nodes == 0:
                continue

            local_map = torch.full((N,), -1, dtype=torch.long, device=device)
            local_map[node_ids] = torch.arange(n_nodes, device=device, dtype=torch.long)

            ei = data.edge_index
            keep = (local_map[ei[0]] >= 0) & (local_map[ei[1]] >= 0)
            keep_idx = keep.nonzero(as_tuple=False).view(-1)
            if keep_idx.numel() == 0:
                H = F.relu(self.gcn1(X0[node_ids], edge_index_local=torch.empty(2,0,device=device,dtype=torch.long), n_nodes=n_nodes))
                H = F.relu(self.gcn2(H,          edge_index_local=torch.empty(2,0,device=device,dtype=torch.long), n_nodes=n_nodes))
                out_node[node_ids] = H
                continue

            src_local = local_map[ei[0, keep_idx]]
            dst_local = local_map[ei[1, keep_idx]]
            edge_local = torch.stack([src_local, dst_local], dim=0)

            H = F.relu(self.gcn1(X0[node_ids], edge_local, n_nodes))
            H = F.relu(self.gcn2(H,            edge_local, n_nodes))
            out_node[node_ids] = H

        if E == 0:
            return torch.empty((0,), device=device)

        u, v = data.edge_index
        U = out_node[u]
        V = out_node[v]
        EA = data.edge_attr if hasattr(data, "edge_attr") and data.edge_attr.numel() > 0 \
             else out_node.new_zeros((E, 5))
        feats = torch.cat([U, V, EA], dim=1)
        logits = self.edge_mlp(feats).squeeze(-1)
        return logits


Using DirectML: privateuseone:0


In [58]:
import torch.nn.functional as F

def _num_graphs_in_batch(node_batch: torch.Tensor) -> int:
    return int(node_batch.max().item()) + 1 if node_batch.numel() else 0

def _edge_batch_from_nodes(node_batch: torch.Tensor, edge_src: torch.Tensor) -> torch.Tensor:
    return node_batch[edge_src] if edge_src.numel() else edge_src.new_zeros((0,), dtype=torch.long)

def _collect_decisions_by_labeled_sources(data, g: int, u: torch.Tensor, edge_batch: torch.Tensor):
    """
    Return list of (cand_idx_tensor, gold_pos_tensor) for EVERY decision node
    whose source node appears on the labeled path. This is orientation-agnostic.
    """
    y = data.y_edge_mask
    lab_eidx = (y == 1).nonzero(as_tuple=False).view(-1)
    lab_eidx = lab_eidx[edge_batch[lab_eidx] == g]
    if lab_eidx.numel() == 0:
        return []

    labeled_sources = torch.unique(u[lab_eidx])

    decisions = []
    for s in labeled_sources.tolist():
        mask = (edge_batch == g) & (u == s)
        cand_idx = mask.nonzero(as_tuple=False).view(-1)
        if cand_idx.numel() < 2:
            continue
        y_here = y[cand_idx]
        gold_pos = (y_here == 1).nonzero(as_tuple=False).view(-1)
        if gold_pos.numel() == 1:
            decisions.append((cand_idx, gold_pos.view(1)))
    return decisions

def train_one_epoch_choice_all(model, loader, optim, device):
    model.train()
    total_loss, total_decisions = 0.0, 0

    for data in loader:
        data = data.to(device)
        logits = model(data)
        if logits.numel() == 0:
            continue

        u = data.edge_index[0]
        node_batch = data.batch
        edge_batch = _edge_batch_from_nodes(node_batch, u)
        num_graphs = _num_graphs_in_batch(node_batch)

        batch_loss = 0.0
        dec_used = 0

        for g in range(num_graphs):
            decisions = _collect_decisions_by_labeled_sources(data, g, u, edge_batch)
            for cand_idx, gold_pos in decisions:
                ce = F.cross_entropy(logits[cand_idx].unsqueeze(0), gold_pos.view(1))
                batch_loss += ce
                dec_used += 1

        if dec_used == 0:
            continue

        batch_loss = batch_loss / dec_used
        optim.zero_grad()
        batch_loss.backward()
        optim.step()

        total_loss += batch_loss.item()
        total_decisions += dec_used

    return total_loss / max(total_decisions, 1)

def _cpu_state_dict(sd):
    return {k: v.detach().cpu().clone() for k, v in sd.items()}

best_ckpt = {
    "epoch": 0,
    "dec_acc": -1.0,
    "bub_acc": -1.0,
    "state_dict": None,
}

def _is_better(bub_acc, dec_acc, best):
    if bub_acc > best["bub_acc"]:
        return True
    if bub_acc == best["bub_acc"] and dec_acc > best["dec_acc"]:
        return True
    return False

@torch.no_grad()
def eval_choice_all(model, loader, device):
    model.eval()
    total_decisions, correct_decisions = 0, 0
    total_bubbles, correct_bubbles = 0, 0

    for data in loader:
        data = data.to(device)
        logits = model(data)
        if logits.numel() == 0:
            continue

        u = data.edge_index[0]
        node_batch = data.batch
        edge_batch = _edge_batch_from_nodes(node_batch, u)
        num_graphs = _num_graphs_in_batch(node_batch)

        for g in range(num_graphs):
            decisions = _collect_decisions_by_labeled_sources(data, g, u, edge_batch)
            if not decisions:
                continue
            total_bubbles += 1

            all_correct = True
            for cand_idx, gold_pos in decisions:
                pred = logits[cand_idx].argmax().item()
                ok = int(pred == gold_pos.item())
                correct_decisions += ok
                total_decisions += 1
                all_correct &= bool(ok)

            correct_bubbles += int(all_correct)

    dec_acc = correct_decisions / max(total_decisions, 1)
    bub_acc = correct_bubbles / max(total_bubbles, 1)
    print(f"[choice-eval] decisions={total_decisions} acc_dec={dec_acc:.3f} | bubbles={total_bubbles} acc_bub={bub_acc:.3f}")
    return dec_acc, bub_acc

In [59]:
import math, time, random
from pathlib import Path
from typing import List, Tuple, Dict, Optional
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt

OUTDIR = Path("sweep_from_paths"); OUTDIR.mkdir(exist_ok=True, parents=True)

SEED = 42
BATCH_SIZE = 64
NUM_WORKERS = 0
EPOCHS = 200
PATIENCE = 200
LR = 1e-3
WEIGHT_DECAY = 0.0
GRAD_CLIP = 1.0
VAL_FRAC = 0.1  # fraction of TRAIN reserved for validation

# Keep grids modest to avoid massive output; expand if you want.
EMBED_GRID = [16, 32]
GCN_HIDDEN_GRID = [16,32, 64, 128]
EDGE_HIDDEN_GRID = [16, 32, 64]
USE_LSTM_OPTS = [False]

PRINT_EVERY = 10  # epoch logging frequency to avoid huge notebook output

print("Using device:", device)

Using device: privateuseone:0


In [60]:
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def labeled_subset(ds):
    from torch.utils.data import Subset
    idx = []
    for i in range(len(ds)):
        d = ds[i]
        lp = int(d.label_path_idx.item()) if hasattr(d, "label_path_idx") else -1
        if lp >= 0:
            idx.append(i)
    return Subset(ds, idx)

def split_train_val(n: int, val_frac: float, seed: int):
    rng = np.random.default_rng(seed)
    order = np.arange(n)
    rng.shuffle(order)
    n_val = int(round(val_frac * n))
    val_idx = order[:n_val]
    tr_idx  = order[n_val:]
    return tr_idx.tolist(), val_idx.tolist()

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def _num_graphs_in_batch(node_batch: torch.Tensor) -> int:
    return int(node_batch.max().item()) + 1 if node_batch.numel() else 0

def _edge_batch_from_nodes(node_batch: torch.Tensor, edge_src: torch.Tensor) -> torch.Tensor:
    return node_batch[edge_src] if edge_src.numel() else edge_src.new_zeros((0,), dtype=torch.long)

def _collect_decisions_by_labeled_sources(data, g: int, u: torch.Tensor, edge_batch: torch.Tensor):
    y = data.y_edge_mask
    lab_eidx = (y == 1).nonzero(as_tuple=False).view(-1)
    lab_eidx = lab_eidx[edge_batch[lab_eidx] == g]
    if lab_eidx.numel() == 0:
        return []
    labeled_sources = torch.unique(u[lab_eidx])
    decisions = []
    for s in labeled_sources.tolist():
        mask = (edge_batch == g) & (u == s)
        cand_idx = mask.nonzero(as_tuple=False).view(-1)
        if cand_idx.numel() < 2:
            continue
        y_here = y[cand_idx]
        gold_pos = (y_here == 1).nonzero(as_tuple=False).view(-1)
        if gold_pos.numel() == 1:
            decisions.append((cand_idx, gold_pos.view(1)))
    return decisions

In [61]:
@torch.no_grad()
def _candidate_sets(data, logits: torch.Tensor):
    """
    Returns list of (cand_idx [M], gold_pos [1], g) for each labeled decision
    in each graph g within the batch.
    """
    u = data.edge_index[0]
    node_batch = data.batch if hasattr(data, "batch") else torch.zeros(data.num_nodes, dtype=torch.long, device=logits.device)
    edge_batch = _edge_batch_from_nodes(node_batch, u)
    num_graphs = _num_graphs_in_batch(node_batch)
    out = []
    for g in range(num_graphs):
        decisions = _collect_decisions_by_labeled_sources(data, g, u, edge_batch)
        for cand_idx, gold_pos in decisions:
            out.append((cand_idx, gold_pos, g))
    return out

def decision_ce_loss_from_batch(logits: torch.Tensor, data, reduction: str = "mean"):
    """
    Cross-entropy over each labeled decision (candidate set).
    """
    if logits.numel() == 0:
        return logits.new_zeros(()), 0
    losses, decisions = [], 0
    u = data.edge_index[0]
    node_batch = data.batch if hasattr(data, "batch") else torch.zeros(data.num_nodes, dtype=torch.long, device=logits.device)
    edge_batch = _edge_batch_from_nodes(node_batch, u)
    num_graphs = _num_graphs_in_batch(node_batch)
    for g in range(num_graphs):
        cands = _collect_decisions_by_labeled_sources(data, g, u, edge_batch)
        for cand_idx, gold_pos in cands:
            cand_logits = logits[cand_idx]  # [M]
            target = gold_pos.view(1)       # [1]
            losses.append(F.cross_entropy(cand_logits.unsqueeze(0), target))
            decisions += 1
    if decisions == 0:
        return logits.new_zeros(()), 0
    loss = torch.stack(losses)
    return (loss.mean() if reduction == "mean" else loss.sum()), decisions

def _ece(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15):
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ece, mce = 0.0, 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (probs >= lo) & (probs < hi if i < n_bins-1 else probs <= hi)
        if not mask.any():
            continue
        conf = probs[mask].mean()
        acc  = labels[mask].mean()
        gap  = abs(conf - acc)
        ece += (mask.mean()) * gap
        mce = max(mce, gap)
    return float(ece), float(mce)

def _mrr_from_logits(logits: torch.Tensor, gold_idx: int):
    ranks = (logits > logits[gold_idx]).sum().item() + 1
    return 1.0 / ranks

@torch.no_grad()
def eval_rich_metrics(model, loader, device) -> Dict[str, float]:
    model.eval()
    dec_correct, dec_total = 0, 0
    bubble_ok = {}  # key: (id(data), g)
    mrrs, nlls, confs, labels, entropies = [], [], [], [], []
    top2_hits, top2_total = 0, 0
    edge_scores, edge_labels = [], []

    for data in loader:
        data = data.to(device)
        logits = model(data)
        if logits.numel() == 0:
            continue

        if hasattr(data, "y_edge_mask"):
            edge_scores.append(logits.detach().cpu().numpy())
            edge_labels.append(data.y_edge_mask.detach().cpu().numpy())

        for cand_idx, gold_pos, g in _candidate_sets(data, logits):
            cand_logits = logits[cand_idx]
            probs = F.softmax(cand_logits, dim=0)
            pred = int(torch.argmax(cand_logits).item())
            gold = int(gold_pos.item())
            ok = int(pred == gold)

            dec_correct += ok
            dec_total += 1

            key = (id(data), int(g))
            bubble_ok.setdefault(key, True)
            bubble_ok[key] = bubble_ok[key] and bool(ok)

            mrrs.append(_mrr_from_logits(cand_logits, gold))
            nlls.append(float(-torch.log(probs[gold] + 1e-12).item()))
            confs.append(float(probs[gold].item()))
            labels.append(float(ok))
            entropies.append(float(-(probs * torch.log(probs + 1e-12)).sum().item()))
            if probs.numel() >= 2:
                top2_total += 1
                top2_hits += int(gold in torch.topk(cand_logits, k=min(2, probs.numel())).indices.tolist())

    bub_total = len(bubble_ok)
    bub_correct = sum(1 for ok in bubble_ok.values() if ok)

    confs_np = np.array(confs, dtype=np.float64)
    labels_np = np.array(labels, dtype=np.float64)
    ece, mce = _ece(confs_np, labels_np, 15)
    brier = float(((confs_np - labels_np) ** 2).mean()) if confs_np.size else float('nan')

    # Edge-level ROC/PR (lightweight approximations)
    if edge_scores and edge_labels:
        scores = np.concatenate(edge_scores).astype(np.float64)
        y_true = np.concatenate(edge_labels).astype(np.int32)
        pos = scores[y_true == 1]; neg = scores[y_true == 0]
        if len(pos) > 0 and len(neg) > 0:
            ranks = scores.argsort().argsort() + 1
            auc = float((ranks[y_true == 1].mean() - (len(pos)+1)/2) / len(neg))
        else:
            auc = float('nan')
        thresholds = np.linspace(scores.min(), scores.max(), num=200) if scores.size else np.array([0.5])
        precisions, recalls = [], []
        P = max((y_true == 1).sum(), 1)
        for t in thresholds:
            y_pred = (scores >= t)
            tp = np.logical_and(y_pred, y_true == 1).sum()
            fp = np.logical_and(y_pred, y_true == 0).sum()
            prec = tp / max(tp + fp, 1)
            rec = tp / P
            precisions.append(prec); recalls.append(rec)
        order = np.argsort(recalls)
        pr_auc = float(np.trapz(np.array(precisions)[order], np.array(recalls)[order])) if len(order) else float('nan')
    else:
        auc, pr_auc = float('nan'), float('nan')

    return {
        "acc_dec": dec_correct / max(dec_total, 1),
        "acc_bub": bub_correct / max(bub_total, 1),
        "top2_acc": top2_hits / max(top2_total, 1),
        "mrr": float(np.mean(mrrs) if mrrs else float('nan')),
        "nll": float(np.mean(nlls) if nlls else float('nan')),
        "brier": brier,
        "ece": ece,
        "mce": mce,
        "roc_auc_edge": auc,
        "pr_auc_edge": pr_auc,
        "mean_entropy": float(np.mean(entropies) if entropies else float('nan')),
        "decisions": dec_total,
        "bubbles": bub_total,
    }

In [62]:
def train_one_epoch(model, loader, device, optimizer, grad_clip: Optional[float] = 1.0):
    model.train()
    total_loss, total_decisions = 0.0, 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(data)
        loss, ndec = decision_ce_loss_from_batch(logits, data, reduction="mean")
        if ndec == 0:
            continue
        loss.backward()
        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        total_loss += float(loss.item()) * ndec
        total_decisions += ndec
    return total_loss / max(total_decisions, 1)

def fit_variant(build_model_fn, train_loader, val_loader, device, out_ckpt: Path,
                epochs=20, lr=1e-3, weight_decay=0.0, patience=5, grad_clip=1.0, seed=42, print_every=5):
    set_seed(seed)
    model = build_model_fn().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val = -1.0
    best_state = None
    waited = 0
    history = []
    for ep in range(1, epochs+1):
        tr_loss = train_one_epoch(model, train_loader, device, opt, grad_clip)
        val_stats = eval_rich_metrics(model, val_loader, device)
        history.append({"epoch": ep, "train_loss": tr_loss, **val_stats})
        metric = val_stats["acc_bub"]
        if (ep % print_every == 0) or (ep == 1):
            print(f"[ep {ep:03d}] train_loss={tr_loss:.4f}  val_bub={val_stats['acc_bub']:.4f}  val_dec={val_stats['acc_dec']:.4f}")
        if metric > best_val:
            best_val = metric
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            waited = 0
        else:
            waited += 1
        if waited >= patience:
            break
    if best_state is None:
        best_state = model.state_dict()
    torch.save(best_state, out_ckpt)
    return best_state, pd.DataFrame(history)

In [63]:
# for pth in TRAIN_PATHS + TEST_PATHS:
#     if not Path(pth).is_file():
#         raise FileNotFoundError(f"Missing: {pth}")

# train_full = HyperbubbleDataset(TRAIN_PATHS)
# test_full  = HyperbubbleDataset(TEST_PATHS)

# # ensure consistent k in each split
# def _check_k(ds):
#     if len(ds) == 0: return
#     s0 = ds[0].seq_tokens.size(1)
#     for i in (len(ds)//2, len(ds)-1):
#         s = ds[i].seq_tokens.size(1)
#         assert s == s0, f"Inconsistent k across samples: {s0} vs {s}"
# _check_k(train_full); _check_k(test_full)

# train_lab = labeled_subset(train_full)
# test_lab  = labeled_subset(test_full)
# if len(train_lab) == 0:
#     raise ValueError("No labeled samples in TRAIN_PATHS.")
# if len(test_lab) == 0:
#     raise ValueError("No labeled samples in TEST_PATHS.")

# # split TRAIN into train/val for early stopping
# tr_idx, va_idx = split_train_val(len(train_lab), VAL_FRAC, SEED)
# from torch.utils.data import Subset
# train_ds = Subset(train_lab, tr_idx)
# val_ds   = Subset(train_lab, va_idx)
# test_ds  = test_lab

# train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
# val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
# test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# print(f"Split -> train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}")

In [64]:
# rows = []
# set_seed(SEED)

# for ed in EMBED_GRID:
#     for gh in GCN_HIDDEN_GRID:
#         for eh in EDGE_HIDDEN_GRID:
#             for ul in USE_LSTM_OPTS:
#                 variant = f"emb{ed}_g{gh}_e{eh}_{'lstm' if ul else 'mean'}"
#                 ckpt = OUTDIR / f"{variant}.pth"
#                 log_csv = OUTDIR / f"{variant}_history.csv"

#                 def build_model():
#                     return HyperbubbleGNN(
#                         vocab_size=5, embed_dim=ed,
#                         gcn_hidden=gh, edge_hidden=eh,
#                         edge_feat_dim=5, use_lstm=ul
#                     )

#                 print(f"\n=== Training {variant} ===")
#                 best_state, hist_df = fit_variant(
#                     build_model_fn=build_model,
#                     train_loader=train_loader,
#                     val_loader=val_loader,
#                     device=device,
#                     out_ckpt=ckpt,
#                     epochs=EPOCHS,
#                     lr=LR,
#                     weight_decay=WEIGHT_DECAY,
#                     patience=PATIENCE,
#                     grad_clip=GRAD_CLIP,
#                     seed=SEED,
#                     print_every=PRINT_EVERY
#                 )
#                 hist_df.to_csv(log_csv, index=False)

#                 model = build_model().to(device)
#                 model.load_state_dict(best_state); model.eval()
#                 stats = eval_rich_metrics(model, test_loader, device)
#                 stats["variant"] = variant
#                 stats["params"] = count_params(model)
#                 rows.append(stats)
#                 print(f"[TEST] {variant}  acc_bub={stats['acc_bub']:.4f}  acc_dec={stats['acc_dec']:.4f}  mrr={stats['mrr']:.4f}")

# df = pd.DataFrame(rows).sort_values(by="acc_bub", ascending=False)
# df_path = OUTDIR / "sweep_results.csv"
# df.to_csv(df_path, index=False)
# print(f"\nSaved: {df_path}")
# df.head(12)

In [65]:
# def plot_bars(df: pd.DataFrame, x_col: str, y_cols: List[str], title: str, ylabel: str, fname: Path):
#     ax = df.set_index(x_col)[y_cols].plot(kind='bar', rot=0)
#     ax.set_title(title); ax.set_ylabel(ylabel)
#     ax.figure.tight_layout(); ax.figure.savefig(str(fname), dpi=200); plt.close(ax.figure)

# def plot_lines(x, series: Dict[str, List[float]], title: str, xlabel: str, ylabel: str, fname: Path):
#     plt.figure()
#     for name, ys in series.items():
#         plt.plot(x, ys, marker='o', label=name)
#     plt.title(title); plt.xlabel(xlabel); plt.ylabel(ylabel)
#     plt.legend(); plt.tight_layout(); plt.savefig(str(fname), dpi=200); plt.close()

# plot_bars(df, "variant", ["acc_bub","acc_dec","mrr"], "Model comparison", "Score", OUTDIR/"cmp_bar.png")

# df2 = df[["params","acc_bub","acc_dec"]].dropna().sort_values("params")
# plot_lines(df2["params"].tolist(),
#            {"Bubble Acc": df2["acc_bub"].tolist(), "Decision Acc": df2["acc_dec"].tolist()},
#            "Accuracy vs #Params", "#Params", "Accuracy", OUTDIR/"acc_vs_params.png")

# # Reliability diagram for best
# best_variant = df.iloc[0]["variant"]
# best_ckpt = OUTDIR / f"{best_variant}.pth"

# def build_from_name(name: str):
#     parts = name.split('_')
#     emb = int(parts[0][3:])
#     g   = int(parts[1][1:])
#     e   = int(parts[2][1:])
#     ul  = ('lstm' in name)
#     m = HyperbubbleGNN(vocab_size=5, embed_dim=emb, gcn_hidden=g, edge_hidden=e, edge_feat_dim=5, use_lstm=ul).to(device)
#     m.load_state_dict(torch.load(best_ckpt, map_location="cpu"))
#     m.eval()
#     return m

# confs_all, labels_all = [], []
# with torch.no_grad():
#     m = build_from_name(best_variant)
#     for data in test_loader:
#         data = data.to(device)
#         logits = m(data)
#         for cand_idx, gold_pos, _g in _candidate_sets(data, logits):
#             probs = F.softmax(logits[cand_idx], dim=0)
#             confs_all.append(float(probs[gold_pos.item()].item()))
#             pred = int(torch.argmax(logits[cand_idx]).item())
#             labels_all.append(float(pred == gold_pos.item()))
# if confs_all:
#     bins = np.linspace(0.0, 1.0, 16)
#     xs, accs = [], []
#     labels_np = np.array(labels_all, dtype=np.float64)
#     confs_np = np.array(confs_all, dtype=np.float64)
#     for i in range(15):
#         lo, hi = bins[i], bins[i+1]
#         mask = (confs_np >= lo) & (confs_np < hi if i<14 else confs_np <= hi)
#         if not mask.any(): 
#             continue
#         xs.append((lo+hi)/2)
#         accs.append(labels_np[mask].mean())
#     plt.figure()
#     plt.plot([0,1],[0,1], linestyle='--')
#     plt.plot(xs, accs, marker='o')
#     plt.title(f"Reliability: {best_variant}"); plt.xlabel("Confidence"); plt.ylabel("Empirical accuracy")
#     plt.tight_layout(); plt.savefig(str(OUTDIR/"reliability_best.png"), dpi=200); plt.close()

# print(f"Artifacts saved in: {OUTDIR.resolve()}")

In [66]:
# from pathlib import Path
# import re, glob, json, numpy as np, pandas as pd
# import torch
# from torch_geometric.loader import DataLoader
# import matplotlib.pyplot as plt

# # Where to look for the datasets (add paths if needed)
# SEARCH_DIRS = [
#     ".", 
#     "other_cov_data",
#     "lower_cov_data",
# ]

# COVERAGES = [10, 20, 30]   # target coverage magnitudes
# GENOME_KEYS = ["ecoli", "klebsiella", "salmonella"]  # expected 3 genomes

# # Sweep (non-LSTM only here)
# EMBED_GRID = [16, 32]
# GCN_HIDDEN_GRID = [32, 64, 128]
# EDGE_HIDDEN_GRID = [16, 32, 64]
# USE_LSTM_OPTS = [False]    # IMPORTANT: no LSTM locally

# # Train/validation/test split within each coverage
# VAL_FRAC = 0.1
# SEED = 42
# PRINT_EVERY = 5

# # I/O
# OUTDIR = Path("cov_sweep_runs"); OUTDIR.mkdir(parents=True, exist_ok=True)
# AGG_PATH = OUTDIR / "cov_all_results.csv"

# print("Device:", device)
# print("Output dir:", OUTDIR.resolve())


In [67]:
# import os, itertools

# def _match_cov_files(coverage: int):
#     """
#     Return dict {genome -> path} for a given coverage.
#     Accept patterns:
#       *_cov{coverage}*_ratio_lt_3.5.jsonl
#       *_cov{coverage}*_ratio_lt_35.jsonl
#       *_cov{coverage}*.jsonl
#     Prefers files with ratio suffix if multiple exist.
#     """
#     patterns = [
#         f"*cov{coverage}*ratio*3.5*.jsonl",  # ratio_lt_3.5.*
#         f"*cov{coverage}*ratio*35*.jsonl",   # ratio_lt_35.*
#         f"*cov{coverage}*.jsonl",            # fallback
#     ]
#     candidates = []
#     for d in SEARCH_DIRS:
#         for pat in patterns:
#             candidates.extend([str(p) for p in Path(d).glob(pat)])
#     # unique by path
#     candidates = list(dict.fromkeys(candidates))

#     def genome_key_from_path(p: str):
#         name = Path(p).name.lower()
#         for g in GENOME_KEYS:
#             if g in name:
#                 return g
#         # fallback: prefix before first underscore
#         return name.split("_")[0]

#     # keep the most specific matches first (ratio name preferred)
#     def score(p: str):
#         name = Path(p).name
#         s = 0
#         if "ratio" in name: s += 2
#         if re.search(r"3[._]?5", name): s += 1
#         return -s  # sort asc by negative score => higher score first

#     candidates.sort(key=score)

#     mapping = {}
#     for p in candidates:
#         g = genome_key_from_path(p)
#         if g in GENOME_KEYS and g not in mapping:
#             mapping[g] = p
#         if len(mapping) == len(GENOME_KEYS):
#             break
#     return mapping

# COV_TO_FILES = {cov: _match_cov_files(cov) for cov in COVERAGES}

# # sanity checks & print
# for cov, mp in COV_TO_FILES.items():
#     print(f"Coverage {cov}:")
#     for g in GENOME_KEYS:
#         print(f"  {g:12s} -> {mp.get(g, 'MISSING')}")
#     if any(g not in mp for g in GENOME_KEYS):
#         raise FileNotFoundError(f"Missing genome files for cov{cov}. Found: {mp}")


In [68]:
# from torch.utils.data import Subset
# import numpy as np

# # Fixed best-variant configuration
# BEST_EMB = 16
# BEST_GCN = 64
# BEST_EDGE = 64
# USE_LSTM = False
# VARIANT = f"emb{BEST_EMB}_g{BEST_GCN}_e{BEST_EDGE}_{'lstm' if USE_LSTM else 'mean'}"

# def set_seed(seed: int):
#     import random
#     random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
#     if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

# def labeled_subset(ds):
#     idx = []
#     for i in range(len(ds)):
#         d = ds[i]
#         lp = int(d.label_path_idx.item()) if hasattr(d, "label_path_idx") else -1
#         if lp >= 0:
#             idx.append(i)
#     return Subset(ds, idx)

# def split_train_val(n: int, val_frac: float, seed: int):
#     rng = np.random.default_rng(seed)
#     order = np.arange(n); rng.shuffle(order)
#     n_val = int(round(val_frac * n))
#     val_idx = order[:n_val]; tr_idx = order[n_val:]
#     return tr_idx.tolist(), val_idx.tolist()

# def count_params(model):
#     return sum(p.numel() for p in model.parameters() if p.requires_grad)

# rows = []
# set_seed(SEED)

# for cov in COVERAGES:
#     paths_map = COV_TO_FILES[cov]  # {genome: path}
#     for fold, test_genome in enumerate(GENOME_KEYS, start=1):
#         test_paths = [paths_map[test_genome]]
#         train_paths = [paths_map[g] for g in GENOME_KEYS if g != test_genome]

#         # Build datasets
#         train_full = HyperbubbleDataset(train_paths)
#         test_full  = HyperbubbleDataset(test_paths)

#         # labeled subsets only
#         train_lab = labeled_subset(train_full)
#         test_lab  = labeled_subset(test_full)
#         if len(train_lab) == 0 or len(test_lab) == 0:
#             raise ValueError(f"No labeled samples for cov{cov}, test={test_genome}")

#         # split train->train/val
#         tr_idx, va_idx = split_train_val(len(train_lab), VAL_FRAC, SEED)
#         train_ds = Subset(train_lab, tr_idx)
#         val_ds   = Subset(train_lab, va_idx)
#         test_ds  = test_lab

#         train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
#         val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
#         test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

#         print(f"\n[cov={cov} | fold={fold} | test={test_genome}] "
#               f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

#         tag = f"cov{cov}_fold{fold}_{test_genome}_{VARIANT}"
#         ckpt = OUTDIR / f"{tag}.pth"
#         log_csv = OUTDIR / f"{tag}_history.csv"

#         def build_model():
#             return HyperbubbleGNN(
#                 vocab_size=5,
#                 embed_dim=BEST_EMB,
#                 gcn_hidden=BEST_GCN,
#                 edge_hidden=BEST_EDGE,
#                 edge_feat_dim=5,
#                 use_lstm=USE_LSTM,
#             )

#         # Train (early stopping inside fit_variant)
#         best_state, hist_df = fit_variant(
#             build_model_fn=build_model,
#             train_loader=train_loader,
#             val_loader=val_loader,
#             device=device,
#             out_ckpt=ckpt,
#             epochs=EPOCHS,
#             lr=LR,
#             weight_decay=WEIGHT_DECAY,
#             patience=PATIENCE,
#             grad_clip=GRAD_CLIP,
#             seed=SEED,
#             print_every=PRINT_EVERY,
#         )
#         hist_df["coverage"] = cov
#         hist_df["fold"] = fold
#         hist_df["test_genome"] = test_genome
#         hist_df["variant"] = VARIANT
#         hist_df.to_csv(log_csv, index=False)

#         # Test
#         model = build_model().to(device)
#         model.load_state_dict(best_state); model.eval()
#         stats = eval_rich_metrics(model, test_loader, device)
#         stats.update({
#             "coverage": cov,
#             "fold": fold,
#             "test_genome": test_genome,
#             "variant": VARIANT,
#             "params": count_params(model),
#         })
#         rows.append(stats)
#         print(f"[TEST] cov={cov} fold={fold} {VARIANT} -> "
#               f"acc_bub={stats['acc_bub']:.4f} acc_dec={stats['acc_dec']:.4f} mrr={stats['mrr']:.4f}")

# # Aggregate & save
# df_cov = pd.DataFrame(rows).sort_values(["coverage","fold","acc_bub"], ascending=[True,True,False])
# df_cov.to_csv(AGG_PATH, index=False)
# print("\nSaved:", AGG_PATH.resolve())
# df_cov.head(12)


In [69]:
# def plot_bars(df: pd.DataFrame, x_col: str, y_cols, title: str, ylabel: str, fname: Path):
#     ax = df.set_index(x_col)[y_cols].plot(kind="bar", rot=0)
#     ax.set_title(title); ax.set_ylabel(ylabel)
#     ax.figure.tight_layout(); ax.figure.savefig(str(fname), dpi=200); plt.close(ax.figure)

# def plot_lines(x, series: Dict[str, list], title: str, xlabel: str, ylabel: str, fname: Path):
#     plt.figure()
#     for name, ys in series.items():
#         plt.plot(x, ys, marker="o", label=name)
#     plt.title(title); plt.xlabel(xlabel); plt.ylabel(ylabel)
#     plt.legend(); plt.tight_layout(); plt.savefig(str(fname), dpi=200); plt.close()

# # 1) Per-coverage: best variant across folds (mean ± std)
# summary = (
#     df_cov
#     .groupby(["coverage","variant"], as_index=False)
#     .agg(acc_bub_mean=("acc_bub","mean"),
#          acc_bub_std=("acc_bub","std"),
#          acc_dec_mean=("acc_dec","mean"),
#          acc_dec_std=("acc_dec","std"),
#          mrr_mean=("mrr","mean"),
#          params=("params","first"))
# )

# # Choose top-5 variants per coverage by acc_bub_mean
# top_variants = (summary.sort_values(["coverage","acc_bub_mean"], ascending=[True,False])
#                       .groupby("coverage").head(5))

# for cov in sorted(df_cov["coverage"].unique()):
#     df_cov_cov = top_variants[top_variants.coverage == cov].copy()
#     if df_cov_cov.empty: 
#         continue
#     fname = OUTDIR / f"cov{cov}_top_variants_bar.png"
#     plot_bars(df_cov_cov, "variant",
#               ["acc_bub_mean","acc_dec_mean","mrr_mean"],
#               f"Coverage {cov}: top variants", "Score", fname)

# # 2) Across coverages: best variant per coverage (by mean acc_bub)
# best_per_cov = (summary.loc[summary.groupby("coverage")["acc_bub_mean"].idxmax()]
#                 .sort_values("coverage"))
# covs = best_per_cov["coverage"].tolist()
# plot_lines(covs,
#            {"Bubble Acc (best)": best_per_cov["acc_bub_mean"].tolist(),
#             "Decision Acc (best)": best_per_cov["acc_dec_mean"].tolist()},
#            "Best-by-coverage accuracy", "Coverage", "Accuracy",
#            OUTDIR / "accuracy_by_coverage_best.png")

# # 3) Params vs accuracy for top-1 per coverage
# plot_lines(best_per_cov["params"].tolist(),
#            {"Bubble Acc": best_per_cov["acc_bub_mean"].tolist(),
#             "Decision Acc": best_per_cov["acc_dec_mean"].tolist()},
#            "Accuracy vs #Params (best per coverage)", "#Params", "Accuracy",
#            OUTDIR / "acc_vs_params_best.png")

# # 4) Optional: save quick pivot for reporting
# pivot = (df_cov.pivot_table(index=["coverage","variant"], values=["acc_bub","acc_dec","mrr"], aggfunc="mean")
#               .reset_index()
#               .sort_values(["coverage","acc_bub"], ascending=[True,False]))
# pivot.to_csv(OUTDIR / "coverage_variant_pivot.csv", index=False)

# print("Plots & pivots saved in:", OUTDIR.resolve())
# pivot.head(10)

In [70]:
# from pathlib import Path
# import re
# import numpy as np
# import pandas as pd
# import torch
# from torch_geometric.loader import DataLoader
# from torch.utils.data import Subset

# # Wyjście
# OUTDIR_AUG = Path("aug_cov20_runs"); OUTDIR_AUG.mkdir(parents=True, exist_ok=True)
# AGG_AUG = OUTDIR_AUG / "aug_vs_plain_cov20.csv"

# # Architektura (stała)
# BEST_EMB = 16
# BEST_GCN = 64
# BEST_EDGE = 64
# USE_LSTM = False
# VARIANT_NAME = f"emb{BEST_EMB}_g{BEST_GCN}_e{BEST_EDGE}_{'lstm' if USE_LSTM else 'mean'}"

# # Gdzie szukać danych
# SEARCH_DIRS = ["other_cov_data", "lower_cov_data", "."]

# # Genomy
# GENOME_KEYS = ["ecoli", "klebsiella", "salmonella"]

# def _discover_triplets_cov20_full_upgraded():
#     """
#     Dla każdego genomu zwróć:
#       base:  *_dataset_cov20_full_upgraded*.jsonl          (bez 'augmented' i bez 'val')  -> trening/test
#       aug:   *_dataset_cov20_full_upgraded*_augmented.jsonl -> trening (augmentacja)
#       val:   *_dataset_cov20_full_upgraded*_val.jsonl       -> walidacja
#     """
#     trip = {g: {"base": None, "aug": None, "val": None} for g in GENOME_KEYS}
#     patterns = [
#         "*_dataset_cov20_full_upgraded*_augmented.jsonl",
#         "*_dataset_cov20_full_upgraded*_val.jsonl",
#         "*_dataset_cov20_full_upgraded*.jsonl",
#     ]
#     cands = []
#     for d in SEARCH_DIRS:
#         p = Path(d)
#         for pat in patterns:
#             cands += [str(x) for x in p.glob(pat)]
#     cands = list(dict.fromkeys(cands))

#     def genome_key(name: str):
#         n = Path(name).name.lower()
#         for g in GENOME_KEYS:
#             if g in n:
#                 return g
#         return None

#     for path in cands:
#         name = Path(path).name.lower()
#         g = genome_key(name)
#         if g is None:
#             continue
#         if "augmented" in name:
#             trip[g]["aug"] = path
#         elif "val" in name:
#             trip[g]["val"] = path
#         else:
#             # base: bez 'augmented' i bez 'val'
#             trip[g]["base"] = path

#     missing = {g: [k for k,v in trip[g].items() if not v] for g in GENOME_KEYS}
#     missing = {g: ms for g,ms in missing.items() if ms}
#     if missing:
#         raise FileNotFoundError(f"Brak wymaganych plików: {missing}\nZnalezione: {trip}")
#     return trip

# def set_seed_all(seed:int):
#     import random
#     random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
#     if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

# def labeled_subset(ds):
#     idx = []
#     for i in range(len(ds)):
#         d = ds[i]
#         lp = int(d.label_path_idx.item()) if hasattr(d, "label_path_idx") else -1
#         if lp >= 0: idx.append(i)
#     return Subset(ds, idx)


In [71]:
# # %% [AUG-RUNNER v2] 3-fold LOYO: walidacja *_val, test z *_full_upgraded, trening base vs augmented

# set_seed_all(SEED)

# trip = _discover_triplets_cov20_full_upgraded()
# rows = []

# for fold, test_genome in enumerate(GENOME_KEYS, start=1):
#     # --- Ścieżki TEST (tylko genom trzymany na test): base full_upgraded
#     test_paths = [trip[test_genome]["base"]]

#     # --- Ścieżki WALIDACJA (zawsze *_val.jsonl) – z pozostałych, trenowanych genomów
#     val_paths = [trip[g]["val"] for g in GENOME_KEYS if g != test_genome]

#     # --- Dwa warianty treningu: plain vs augmented (z pozostałych genomów)
#     for use_aug in [False, True]:
#         if use_aug:
#             train_paths = [trip[g]["aug"]  for g in GENOME_KEYS if g != test_genome]
#             label_txt = "aug"
#         else:
#             train_paths = [trip[g]["base"] for g in GENOME_KEYS if g != test_genome]
#             label_txt = "plain"

#         # Zbiory danych
#         train_full = HyperbubbleDataset(train_paths)
#         val_full   = HyperbubbleDataset(val_paths)
#         test_full  = HyperbubbleDataset(test_paths)

#         # Tylko etykietowane próbki
#         train_ds = labeled_subset(train_full)
#         val_ds   = labeled_subset(val_full)
#         test_ds  = labeled_subset(test_full)

#         if len(train_ds) == 0 or len(val_ds) == 0 or len(test_ds) == 0:
#             raise ValueError(f"Brak etykiet (fold={fold}, test={test_genome}, {label_txt})")

#         train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
#         val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
#         test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

#         print(f"\n[fold {fold} | test={test_genome} | {label_txt}] "
#               f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

#         tag = f"cov20_fold{fold}_{test_genome}_{label_txt}_{VARIANT_NAME}"
#         ckpt = OUTDIR_AUG / f"{tag}.pth"
#         log_csv = OUTDIR_AUG / f"{tag}_hist.csv"

#         def build_model():
#             return HyperbubbleGNN(
#                 vocab_size=5,
#                 embed_dim=BEST_EMB,
#                 gcn_hidden=BEST_GCN,
#                 edge_hidden=BEST_EDGE,
#                 edge_feat_dim=5,
#                 use_lstm=USE_LSTM,
#             )

#         # Trening (early stopping na walidacji *_val)
#         best_state, hist_df = fit_variant(
#             build_model_fn=build_model,
#             train_loader=train_loader,
#             val_loader=val_loader,      # <- używamy osobnego zbioru *_val
#             device=device,
#             out_ckpt=ckpt,
#             epochs=EPOCHS,
#             lr=LR,
#             weight_decay=WEIGHT_DECAY,
#             patience=PATIENCE,
#             grad_clip=GRAD_CLIP,
#             seed=SEED,
#             print_every=5,
#         )

#         # Historia -> CSV
#         hist_df["coverage"] = 20
#         hist_df["fold"] = fold
#         hist_df["test_genome"] = test_genome
#         hist_df["variant"] = VARIANT_NAME
#         hist_df["augmented_train"] = use_aug
#         hist_df.to_csv(log_csv, index=False)

#         # Test na *_full_upgraded genomu trzymanego na test
#         model = build_model().to(device)
#         model.load_state_dict(best_state); model.eval()
#         stats = eval_rich_metrics(model, test_loader, device)
#         stats.update({
#             "coverage": 20,
#             "fold": fold,
#             "test_genome": test_genome,
#             "variant": VARIANT_NAME,
#             "augmented_train": use_aug,
#             "params": sum(p.numel() for p in model.parameters() if p.requires_grad),
#         })
#         rows.append(stats)

#         print(f"  [TEST] {label_txt}: acc_bub={stats['acc_bub']:.4f}  acc_dec={stats['acc_dec']:.4f}  mrr={stats['mrr']:.4f}")

# # Zapis zbiorczego CSV (pod to masz już komórkę agregującą w PL)
# df_aug2 = pd.DataFrame(rows).sort_values(["fold","augmented_train","acc_bub"], ascending=[True, True, False])
# df_aug2.to_csv(AGG_AUG, index=False)

# print("\nZapisano:")
# print(f"  - wyniki zbiorcze: {AGG_AUG.resolve()}")
# print(f"  - checkpointy i historie: {OUTDIR_AUG.resolve()}")


In [72]:
# %% [INSIDE-GENOME SWEEP] train/test within the same genome (80/20), best config only (emb16_g64_e64_mean, no LSTM, no augment)

import numpy as np
from pathlib import Path
from typing import Dict, List
import torch
from torch.utils.data import Subset
from torch_geometric.loader import DataLoader
import pandas as pd
import matplotlib.pyplot as plt

OUTDIR_IN = Path("inside_genome_runs"); OUTDIR_IN.mkdir(parents=True, exist_ok=True)
INSIDE_CSV = OUTDIR_IN / "inside_genome_results.csv"

# Best-fixed architecture (no LSTM, no augmentation)
BEST_EMB, BEST_GCN, BEST_EDGE = 16, 32, 16
USE_LSTM = False
VARIANT_NAME = f"emb{BEST_EMB}_g{BEST_GCN}_e{BEST_EDGE}_{'lstm' if USE_LSTM else 'mean'}"

# Where to look for per-genome cov20 files; we target "full_upgraded ... ratio_lt_35"
SEARCH_DIRS = ["other_cov_data", "lower_cov_data", "."]
GENOME_KEYS = ["ecoli", "klebsiella", "salmonella"]  # extend if you have more

TRAIN_FRAC = 0.8   # of labeled samples in the genome
VAL_FRAC_IN_TRAIN = 0.1  # from the train part (for early stopping)

def set_seed_all(seed:int):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def labeled_subset(ds):
    idx = []
    for i in range(len(ds)):
        d = ds[i]
        lp = int(d.label_path_idx.item()) if hasattr(d, "label_path_idx") else -1
        if lp >= 0: idx.append(i)
    return Subset(ds, idx)

def _discover_cov20_full_upgraded_ratio35() -> Dict[str, str]:
    """
    Find exactly one base file per genome:
      *_dataset_cov20_full_upgraded*ratio_lt_35.jsonl
    Excludes *_val* and *_augmented* to ensure we use the base dataset.
    """
    cands: List[str] = []
    for d in SEARCH_DIRS:
        p = Path(d)
        cands += [str(x) for x in p.glob("*_dataset_cov20_full_upgraded*ratio_lt_35*.jsonl")]
    # drop duplicates, filter out validation/augmented
    seen = []
    filt = []
    for c in cands:
        name = Path(c).name.lower()
        if ("_val" in name) or ("augmented" in name):
            continue
        if c not in seen:
            seen.append(c); filt.append(c)

    def genome_key(name: str):
        n = Path(name).name.lower()
        for g in GENOME_KEYS:
            if g in n:
                return g
        return None

    m: Dict[str, str] = {}
    for path in filt:
        g = genome_key(path)
        if g is None: 
            continue
        # prefer first match; warn if multiple but keep first
        if g not in m:
            m[g] = path
    missing = [g for g in GENOME_KEYS if g not in m]
    if missing:
        raise FileNotFoundError(f"Missing cov20 full_upgraded ratio_lt_35 files for: {missing}\nFound: {m}")
    return m

def split_indices(n: int, train_frac: float, seed: int):
    rng = np.random.default_rng(seed)
    order = np.arange(n)
    rng.shuffle(order)
    n_train = int(round(train_frac * n))
    train_idx = order[:n_train].tolist()
    test_idx  = order[n_train:].tolist()
    return train_idx, test_idx

def carve_val_from_train(train_idx: List[int], val_frac: float, seed: int):
    if len(train_idx) == 0 or val_frac <= 0:
        return train_idx, []
    rng = np.random.default_rng(seed + 123)
    order = np.array(train_idx, dtype=int)
    rng.shuffle(order)
    n_val = int(round(val_frac * len(order)))
    val_idx = order[:n_val].tolist()
    tr_idx  = order[n_val:].tolist()
    return tr_idx, val_idx

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

set_seed_all(SEED)

genome_to_file = _discover_cov20_full_upgraded_ratio35()
print("Discovered files per genome:")
for g, pth in genome_to_file.items():
    print(f"  - {g}: {pth}")

rows = []

for genome in GENOME_KEYS:
    base_path = genome_to_file[genome]

    # Single-genome dataset
    full_ds = HyperbubbleDataset([base_path])
    lab_ds  = labeled_subset(full_ds)
    if len(lab_ds) < 5:
        raise ValueError(f"Too few labeled samples for genome={genome}: {len(lab_ds)}")

    # 80/20 split on labeled samples
    tr_idx_all, te_idx = split_indices(len(lab_ds), TRAIN_FRAC, SEED)
    tr_idx, va_idx = carve_val_from_train(tr_idx_all, VAL_FRAC_IN_TRAIN, SEED)

    train_ds = Subset(lab_ds, tr_idx)
    val_ds   = Subset(lab_ds, va_idx)
    test_ds  = Subset(lab_ds, te_idx)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    print(f"\n[inside-genome] {genome}  train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

    tag = f"inside_{genome}_{VARIANT_NAME}"
    ckpt = OUTDIR_IN / f"{tag}.pth"
    hist_csv = OUTDIR_IN / f"{tag}_history.csv"

    def build_model():
        return HyperbubbleGNN(
            vocab_size=5,
            embed_dim=BEST_EMB,
            gcn_hidden=BEST_GCN,
            edge_hidden=BEST_EDGE,
            edge_feat_dim=5,
            use_lstm=USE_LSTM,
        )

    best_state, hist_df = fit_variant(
        build_model_fn=build_model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        out_ckpt=ckpt,
        epochs=EPOCHS,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        patience=PATIENCE,
        grad_clip=GRAD_CLIP,
        seed=SEED,
        print_every=5,
    )
    hist_df["genome"] = genome
    hist_df["variant"] = VARIANT_NAME
    hist_df["train_frac"] = TRAIN_FRAC
    hist_df["val_frac_in_train"] = VAL_FRAC_IN_TRAIN
    hist_df.to_csv(hist_csv, index=False)

    model = build_model().to(device)
    model.load_state_dict(best_state); model.eval()
    stats = eval_rich_metrics(model, test_loader, device)
    stats.update({
        "genome": genome,
        "variant": VARIANT_NAME,
        "train_frac": TRAIN_FRAC,
        "val_frac_in_train": VAL_FRAC_IN_TRAIN,
        "params": count_params(model),
        "n_train": len(train_ds),
        "n_val": len(val_ds),
        "n_test": len(test_ds),
    })
    rows.append(stats)

# Save combined CSV
df_inside = pd.DataFrame(rows).sort_values("genome")
df_inside.to_csv(INSIDE_CSV, index=False)
print("\nSaved:", INSIDE_CSV.resolve())
display(df_inside.head())

# Quick plot: acc_bub/acc_dec per genome
plt.figure(figsize=(6,4))
dfp = df_inside.set_index("genome")[["acc_bub","acc_dec"]]
ax = dfp.plot(kind="bar", rot=0)
ax.set_ylabel("Accuracy")
ax.set_title("Inside-genome: acc_bub / acc_dec (train 80% — test 20%)")
ax.figure.tight_layout()
plot_path = OUTDIR_IN / "inside_genome_bar.png"
ax.figure.savefig(plot_path, dpi=200)
plt.close(ax.figure)
print("Saved plot:", plot_path.resolve())


Discovered files per genome:
  - ecoli: lower_cov_data\ecoli_dataset_cov20_full_upgraded_ratio_lt_35.jsonl
  - klebsiella: lower_cov_data\klebsiella_dataset_cov20_full_upgraded_ratio_lt_35.jsonl
  - salmonella: lower_cov_data\salmonella_dataset_cov20_full_upgraded_ratio_lt_35.jsonl

[inside-genome] ecoli  train=247  val=28  test=69
[ep 001] train_loss=0.7582  val_bub=0.4286  val_dec=0.4545
[ep 005] train_loss=0.6994  val_bub=0.5714  val_dec=0.5758
[ep 010] train_loss=0.6863  val_bub=0.5357  val_dec=0.5455
[ep 015] train_loss=0.6866  val_bub=0.4643  val_dec=0.4242
[ep 020] train_loss=0.6850  val_bub=0.5357  val_dec=0.5455
[ep 025] train_loss=0.6853  val_bub=0.5357  val_dec=0.5758
[ep 030] train_loss=0.6846  val_bub=0.5000  val_dec=0.4545
[ep 035] train_loss=0.6832  val_bub=0.5000  val_dec=0.5455
[ep 040] train_loss=0.6844  val_bub=0.5000  val_dec=0.5152
[ep 045] train_loss=0.6760  val_bub=0.5357  val_dec=0.5152
[ep 050] train_loss=0.6757  val_bub=0.5714  val_dec=0.5758
[ep 055] train_lo

,acc_dec,acc_bub,top2_acc,mrr,nll,brier,ece,mce,roc_auc_edge,pr_auc_edge,...,decisions,bubbles,genome,variant,train_frac,val_frac_in_train,params,n_train,n_val,n_test
0,0.480000,0.449275,1.0,0.740000,0.714086,0.217552,0.110676,0.445597,0.403948,0.204312,...,75,69,ecoli,emb16_g32_e16_mean,0.8,0.1,2849,247,28,69
1,0.590164,0.596491,1.0,0.795082,0.656383,0.166884,0.304588,0.437534,0.355227,0.230948,...,61,57,klebsiella,emb16_g32_e16_mean,0.8,0.1,2849,203,23,57
2,0.557692,0.555556,1.0,0.778846,0.694507,0.202731,0.241673,0.448587,0.388296,0.193474,...,104,99,salmonella,emb16_g32_e16_mean,0.8,0.1,2849,358,40,99


Saved plot: C:\Users\Przemek\Desktop\Inzynierka stuff\own_assembler\dna-assembly-bsc\bubble_resolution_gnn\inside_genome_runs\inside_genome_bar.png


<Figure size 600x400 with 0 Axes>